# Fraud Detection Demo — Training Notebook

**Dataset:** Sparkov credit-card transaction simulation ([Kaggle: kartik2112/fraud-detection](https://www.kaggle.com/datasets/kartik2112/fraud-detection)) — ~1.9M human-readable transactions, ~0.6% fraud.

**Pipeline:** load → EDA → feature engineering → XGBoost → evaluation → SHAP → LLM-style report.

Run from the **repo root** so `src` imports resolve.

In [ ]:
# If running in Colab / a fresh venv, uncomment:
# !pip install -q pandas numpy scikit-learn xgboost shap duckdb pyarrow matplotlib joblib kagglehub

In [ ]:
import os, sys, json
# Resolve repo root robustly (works from repo root AND from notebooks/)
root = os.getcwd()
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'src')):
    root = os.path.dirname(root)
os.chdir(root)
sys.path.insert(0, root)

import pandas as pd
import matplotlib.pyplot as plt

# --- Load data -------------------------------------------------------------
# Real dataset:  python -m scripts.download_data   (then concatenate train+test)
# Fallback:      synthetic Sparkov-schema sample generated on the fly
path = 'data/raw/all.csv'
if not os.path.exists(path):
    print('real dataset not found — generating synthetic sample instead')
    from scripts.make_sample_data import generate
    path = generate(n=30000)
df = pd.read_csv(path)
print(f'{len(df):,} rows | fraud rate {df.is_fraud.mean():.4%}')
df.head(3)

## 1. Quick EDA — why fraud detection is hard (class imbalance)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['amt_log'] = __import__('numpy').log1p(df['amt'])
for label, sub in df.groupby('is_fraud'):
    axes[0].hist(sub['amt_log'], bins=60, alpha=0.6, density=True,
                 label='fraud' if label else 'legit')
axes[0].set_title('Amount distribution (log)'); axes[0].legend()

hr = pd.to_datetime(df['trans_date_trans_time']).dt.hour
df.assign(hour=hr).groupby(['hour','is_fraud']).size().unstack(fill_value=0)\
  .pipe(lambda d: (d[1]/d.sum(axis=1))).plot(ax=axes[1], color='crimson')
axes[1].set_title('Fraud rate by hour of day')

df.groupby('category')['is_fraud'].mean().sort_values().plot.barh(ax=axes[2])
axes[2].set_title('Fraud rate by merchant category')
plt.tight_layout(); plt.show()

## 2. Feature engineering

Behavioural signals: amount z-score vs. the customer's own history, distance from home to merchant (haversine), night-time flag, 24h transaction velocity, plus category one-hots.

In [ ]:
from src.features import engineer, BASE_FEATURES
fe, _ = engineer(df.sample(min(5000, len(df)), random_state=1), fit=True)
fe[BASE_FEATURES].describe().T[['mean','min','max']]

## 3. Train XGBoost (time-based split — last 20% of the timeline is held out)

In [ ]:
from src.train import train
metrics = train(df, out_dir='artifacts')
print(json.dumps(metrics, indent=2))

## 4. Global explainability — SHAP feature importance

In [ ]:
from IPython.display import Image
Image('artifacts/shap_summary.png')

## 5. Score new transactions + per-flag SHAP drivers

In [ ]:
from src.scorer import load_bundle, score, shap_explain

bundle = load_bundle('artifacts/model.joblib')
sample = df.sample(2000, random_state=7)
proba, pred = score(sample, bundle)
flagged = sample[pred == 1].copy()
flagged['fraud_probability'] = proba[pred == 1]
print(f'{len(flagged)} flagged of {len(sample)} '
      f'(true frauds caught: {int(flagged.is_fraud.sum())}/{int(sample.is_fraud.sum())})')
flagged[['trans_date_trans_time','first','last','merchant','amt','fraud_probability','is_fraud']].head(10)

## 6. LLM-style investigation report

Without an API key this renders the **offline template** (same data path). Set `LLM_PROVIDER=gemini` + `GEMINI_API_KEY` (or `openai` + `OPENAI_API_KEY`) for the real narrative report.

In [ ]:
from src.llm_analyst import generate_report

single = pd.DataFrame([flagged.iloc[0]]).drop(
    columns=['fraud_probability'], errors='ignore')
p, _ = score(single, bundle)
sf = shap_explain(single, bundle)[0]
hist = df[df.cc_num == single.iloc[0].cc_num]
summary = (f'{len(hist)} transactions; avg ${hist.amt.mean():,.2f}, '
           f'max ${hist.amt.max():,.2f}')

from IPython.display import Markdown
Markdown(generate_report(single.iloc[0].to_dict(), summary, sf, float(p[0])))

---
## Next steps
1. `python -m scripts.prepare_app_data --data <your csv>` → parquet for the app
2. `streamlit run app/streamlit_app.py` — local check
3. Deploy to **Cloud Run**: see README.md (Dockerfile + `deploy_cloudrun.sh` included)
4. Wire up Gemini (`LLM_PROVIDER=gemini`) for live LLM reports & NL→SQL